# RAGAS evaluation for a Bedrock Managed Knowledge Base

This notebook grades your Managed KB's answers with RAGAS, an open-source tool for scoring RAG systems. It rates four things, using one model as a judge and one model for embeddings:

| Metric | What it checks | What it needs |
|--------|----------------|---------------|
| Faithfulness | Does the answer stick to the retrieved text, with no made-up facts? | answer + contexts |
| Answer relevancy | Does the answer actually respond to the question? | question + answer |
| Context precision | Are the retrieved chunks the useful ones? | question + contexts + ground truth |
| Context recall | Did retrieval find everything it needed? | contexts + ground truth |

### How this is different from the other eval notebooks here

| Notebook | How it works | Works for a Managed KB? |
|----------|--------------|-------------------------|
| `02-bedrock-evaluation-job.ipynb` | Bedrock Evaluation Job (`CreateEvaluationJob`) | Retrieval only |
| `03-agentcore-evaluation-for-managed-kb.ipynb` | AgentCore Evaluations (scores OTEL trace spans) | Yes |
| This notebook | You retrieve and generate the answers yourself, then score them with RAGAS | Yes |

> Why RAGAS works for a Managed KB: it never calls a KB evaluation API. You build the `(question, answer, contexts, ground_truth)` records yourself, and RAGAS scores them. We generate the answers with `AgenticRetrieveStream`, which Managed KBs support, instead of `RetrieveAndGenerate`, which they do not. So the whole flow is safe to run against a Managed KB.

> Note on `utils/evaluation.py`: it has a `BMKBEvaluator.evaluate_with_ragas()` method, but that method calls `RetrieveAndGenerate`, which Managed KBs do not support. This notebook uses `AgenticRetrieveStream` instead, so prefer this flow for a Managed KB.

## Prerequisites

- A Managed KB that already has documents ingested (create one with `01-getting-started/01-create-bmkb-s3.ipynb`)
- AWS credentials with Bedrock (`bedrock-agent-runtime`) permissions
- Model access turned on for the generation, judge, and embedding models
- A question-and-answer dataset. Make one with `01-qna-generation-from-pdf.ipynb`, or use the small built-in sample below.
- Install `ragas` and `datasets` (plus this repo's `requirements.txt`)
- Kernel: pick `Python 3`

In [ ]:
%pip install --upgrade pip --quiet
# RAGAS 0.1.21 needs the older langchain stack (langchain-core below 0.3), so pin it.
%pip install --quiet \
    "langchain-core==0.2.43" \
    "langchain==0.2.16" \
    "langchain-community==0.2.17" \
    "langchain-text-splitters==0.2.4" \
    "langchain-aws==0.1.18" \
    "ragas==0.1.21" \
    "datasets>=2.14.0"
# AgenticRetrieveStream needs boto3 1.43 or newer. langchain-aws pins boto3 below
# 1.35, but that pin does not matter here because we pass in our own client, so
# bump boto3 back up afterward.
%pip install --upgrade --quiet "boto3>=1.43.0" "botocore>=1.43.0" "s3transfer>=0.13.0" 

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Papermill parameters.
# This cell is tagged 'parameters' so papermill can set these for a headless run.
# In a normal Jupyter session, leave kb_id = None and it will be read back via %store.
kb_id = None
max_questions = 10        # how many questions to score in one run
num_results = 5           # how many chunks to retrieve per question

## Step 1: Configuration

In [ ]:
import boto3
import json
import time
import sys
from botocore.config import Config

sys.path.insert(0, "../..")

session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

# Use longer timeouts, since agentic retrieval plus generation can take a while.
runtime_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 3})
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=region, config=runtime_config)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region, config=runtime_config)

# ── Managed KB ────────────────────────────────────────────────────────
# kb_id comes from the parameters cell above. If it was not set there (a normal
# interactive run), read back the one that
# 01-getting-started/01-create-bmkb-s3.ipynb saved with %store.
if kb_id is None:
    try:
        %store -r kb_id
    except Exception:
        kb_id = None
if not kb_id:
    raise ValueError(
        "kb_id is not set. Either run 01-getting-started/01-create-bmkb-s3.ipynb "
        "first (it saves kb_id with `%store`), or pass kb_id as a papermill parameter."
    )

# Check num_results now. The API allows 1 to 100. Checking here gives a clear
# message instead of a confusing per-question error later in Step 3.
if not isinstance(num_results, int) or not (1 <= num_results <= 100):
    raise ValueError(f'num_results must be an integer from 1 to 100, got {num_results!r}')

print(f'KB ID: {kb_id}')

# ── Models ────────────────────────────────────────────────────────────
# Pick the inference-profile prefix for the current region. If the region is not
# one where Managed KB and these inference profiles exist, stop with a clear
# message instead of building a mismatched ARN (for example a 'us.' profile in
# ca-central-1) that would fail later inside AgenticRetrieveStream.
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region and region.startswith(k)), None)
if cris_prefix is None:
    raise ValueError(
        f'Region {region!r} has no matching inference-profile prefix. '
        'This notebook expects a us, eu, or ap region, where Managed KB and the '
        'Claude and Titan inference profiles are available. Switch regions, or '
        'edit region_prefix_map and the model IDs for your region.'
    )

# Generation model for AgenticRetrieveStream (this needs an inference-profile ARN).
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

# The judge model that RAGAS uses, and the embedding model.
judge_model_id = f'{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'
embedding_model_id = 'amazon.titan-embed-text-v2:0'

print(f'Region:      {region}')
print(f'Account:     {account_id}')
print(f'KB ID:       {kb_id}')
print(f'Generation:  {generation_model_arn}')
print(f'Judge:       {judge_model_id}')
print(f'Embeddings:  {embedding_model_id}')

## Step 2: Load the question-and-answer dataset

RAGAS needs a `question` and a `ground_truth` (the reference answer) for each example. It scores context precision and context recall by comparing against that ground truth.

The cell below first looks for the dataset that `01-qna-generation-from-pdf.ipynb` writes (`evaluation_data/rag_dataset_prompt_with_gt.jsonl`). If that file is not there, it uses a small built-in Octank Financial sample so the notebook still runs start to finish.

In [ ]:
import os

def load_qna_from_jsonl(path):
    """Read the conversationTurns JSONL that notebook 01 writes and return (questions, ground_truths).

    If a line is broken (bad JSON or a missing key), skip it and print a warning
    instead of stopping the whole run. That way one bad row in your dataset does
    not crash the notebook.
    """
    questions, ground_truths = [], []
    skipped = 0
    with open(path) as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                turn = json.loads(line)['conversationTurns'][0]
                q = turn['prompt']['content'][0]['text']
                refs = turn.get('referenceResponses') or []
                gt = refs[0]['content'][0]['text'] if refs else ''
            except (json.JSONDecodeError, KeyError, IndexError, TypeError) as e:
                skipped += 1
                print(f'  Skipping bad line {lineno}: {type(e).__name__}')
                continue
            if q and gt:
                questions.append(q)
                ground_truths.append(gt)
    if skipped:
        print(f'  Skipped {skipped} bad line(s)')
    return questions, ground_truths

dataset_path = 'evaluation_data/rag_dataset_prompt_with_gt.jsonl'

if os.path.exists(dataset_path):
    questions, ground_truths = load_qna_from_jsonl(dataset_path)
    print(f'Loaded {len(questions)} Q&A pairs from {dataset_path}')
else:
    print(f'{dataset_path} not found. Using the built-in Octank Financial sample.')
    print('(To make a real dataset, run 01-qna-generation-from-pdf.ipynb.)')
    # These ground-truth answers come from the real Octank Financial 10K PDF
    # (synthetic_dataset/octank_financial_10K.pdf). Getting them right matters:
    # context_precision and context_recall are scored against the ground truth, so
    # a reference that does not match the document gives low retrieval scores even
    # when retrieval is actually working.
    sample = [
        {
            'question': "What are Octank Financial's primary business segments and product offerings?",
            'ground_truth': 'Octank Financial offers retail financial services (personal loans, business loans, credit cards, and investment services) and institutional financial services (investment banking, wealth management, and asset management).',
        },
        {
            'question': "What was Octank Financial's total revenue in 2021 and how did it change year over year?",
            'ground_truth': 'Octank Financial reported total revenue of $5,000 million in 2021, an increase of $500 million (about 11.1%) from $4,500 million in 2020.',
        },
        {
            'question': 'What are the key risk factors facing Octank Financial?',
            'ground_truth': 'Key risks include economic and industry factors such as changes in interest rates, inflation, and global economic conditions; dependence on key personnel; competitive pressure; and regulatory and operational risks.',
        },
        {
            'question': "What drove Octank Financial's revenue growth?",
            'ground_truth': 'Revenue growth was driven by strong performance in the core business segments together with the acquisition of several smaller companies.',
        },
    ]
    questions = [s['question'] for s in sample]
    ground_truths = [s['ground_truth'] for s in sample]

# Check the cap. max_questions must be a positive whole number. A bad value would
# otherwise quietly give you an empty or backwards list (a negative slice takes
# from the end), which then fails later with a confusing RAGAS error.
if not isinstance(max_questions, int) or max_questions < 1:
    raise ValueError(f'max_questions must be a positive integer, got {max_questions!r}')

# Trim to max_questions to keep the run short. Raise it (in the parameters cell)
# for a fuller evaluation.
questions = questions[:max_questions]
ground_truths = ground_truths[:max_questions]
assert len(questions) == len(ground_truths)
if not questions:
    raise ValueError('No questions to evaluate. Check your dataset file or the built-in sample.')
print(f'Evaluating {len(questions)} questions')

## Step 3: Generate answers and retrieve contexts

For each question we call `AgenticRetrieveStream` with `generateResponse=True`. From that one response we keep two things:

- the generated answer, which RAGAS uses for faithfulness and answer relevancy
- the retrieved chunks, which RAGAS uses for faithfulness and for context precision and recall

Taking both from the same call keeps the chunks matched to the answer they produced, which is what faithfulness is meant to measure.

> We do not use `RetrieveAndGenerate` here because Managed KBs do not support it. `AgenticRetrieveStream` is the way to generate answers from a Managed KB.

In [ ]:
def answer_and_contexts(query):
    """Run AgenticRetrieveStream and return (answer, [context_texts])."""
    resp = bedrock_agent_runtime.agentic_retrieve_stream(
        messages=[{'role': 'user', 'content': {'text': query}}],
        retrievers=[{
            'configuration': {
                'knowledgeBase': {
                    'knowledgeBaseId': kb_id,
                    'retrievalOverrides': {'maxNumberOfResults': num_results},
                }
            }
        }],
        agenticRetrieveConfiguration={
            'foundationModelConfiguration': {
                'bedrockFoundationModelConfiguration': {
                    'modelConfiguration': {'modelArn': generation_model_arn}
                },
                'type': 'BEDROCK_FOUNDATION_MODEL',
            },
            'foundationModelType': 'CUSTOM',
            'maxAgentIteration': 3,
            'rerankingModelType': 'MANAGED',
        },
        generateResponse=True,
    )

    # Usually the answer arrives in the final 'result' event, under
    # generatedResponse. As a backup, also collect any streamed 'responseEvent'
    # text pieces. This matches how
    # utils/managed_knowledge_base.agentic_retrieve_stream reads the stream.
    results, gen_resp, text_chunks = [], None, []
    for event in resp['stream']:
        if 'responseEvent' in event:
            text_chunks.append(event['responseEvent'].get('text', ''))
        elif 'result' in event:
            results = event['result'].get('results', [])
            gen_resp = event['result'].get('generatedResponse')

    answer = ''
    if gen_resp and gen_resp.get('answer'):
        answer = gen_resp['answer']
    elif text_chunks:
        answer = ''.join(text_chunks)

    contexts = [r.get('content', {}).get('text', '') for r in results]
    contexts = [c for c in contexts if c]
    return answer, contexts

answers, contexts_list = [], []
for i, q in enumerate(questions, 1):
    try:
        ans, ctx = answer_and_contexts(q)
    except Exception as e:
        print(f'  [{i}/{len(questions)}] error: {e}')
        ans, ctx = '', []
    answers.append(ans)
    contexts_list.append(ctx)
    print(f'  [{i}/{len(questions)}] answer chars={len(ans)}, chunks={len(ctx)}')
    time.sleep(2)  # small pause so we do not get throttled

print(f'\nGenerated {sum(1 for a in answers if a)} / {len(questions)} answers')

## Step 4: Set up RAGAS with Bedrock

RAGAS needs a model to act as the judge and a model for embeddings. We wrap two Bedrock models with `langchain-aws` and pass them in. The `datasets.Dataset` we build needs four matching columns: `question`, `answer`, `contexts`, and `ground_truth`.

In [ ]:
from langchain_aws.chat_models.bedrock import ChatBedrock
from langchain_aws.embeddings.bedrock import BedrockEmbeddings
from datasets import Dataset

judge_llm = ChatBedrock(
    model_id=judge_model_id,
    client=bedrock_runtime,
    model_kwargs={'max_tokens': 4096, 'temperature': 0.0},
)
embeddings = BedrockEmbeddings(model_id=embedding_model_id, client=bedrock_runtime)

# Drop any row where generation did not give us a usable answer and some contexts,
# since RAGAS cannot score those. The .strip() check makes sure a whitespace-only
# answer (like '\n') does not sneak through.
rows = [
    {'question': q, 'answer': a, 'contexts': c, 'ground_truth': gt}
    for q, a, c, gt in zip(questions, answers, contexts_list, ground_truths)
    if a and a.strip() and c
]
dropped = len(questions) - len(rows)
if dropped:
    print(f'Dropped {dropped} row(s) with no answer/contexts')

# Stop early if every row was dropped. That means generation failed for all of
# them, usually because the kb_id is wrong or not synced, the generation model is
# not turned on in this account or region, or requests are being throttled. If we
# did not stop here, RAGAS would fail with a confusing
# "Dataset feature 'question' should be of type string" message.
if not rows:
    raise RuntimeError(
        f'Nothing to score. All {len(questions)} question(s) came back empty in '
        'Step 3. Common causes: the kb_id does not exist or is not synced, the '
        'generation model is not turned on in this account or region, or requests '
        'are being throttled. Look at the per-question errors printed in Step 3.'
    )

dataset = Dataset.from_dict({
    'question':     [r['question'] for r in rows],
    'answer':       [r['answer'] for r in rows],
    'contexts':     [r['contexts'] for r in rows],
    'ground_truth': [r['ground_truth'] for r in rows],
})
print(f'RAGAS dataset: {len(dataset)} examples')

## Step 5: Run the RAGAS evaluation

Score all four metrics. Each one asks the judge model (and, for relevancy and precision, the embedding model) about every example, so expect several model calls per question.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=embeddings,
)

print('RAGAS scores (dataset average):')
print(results)

## Step 6: Look at the per-question scores

Put the scores in a DataFrame so you can read them per question, and save a CSV. Every score runs from 0 to 1, where higher is better. Watch for single questions that pull an average down. Those usually point to a retrieval gap or a made-up answer.

In [ ]:
import pandas as pd

df = results.to_pandas()

metric_cols = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
metric_cols = [c for c in metric_cols if c in df.columns]

# Print each average along with how many rows were NaN. RAGAS returns NaN for a
# metric when the judge or embedding call failed, or the metric did not apply. A
# plain .mean() quietly skips those, so we show the count to keep the average honest.
print('Average scores (NaN means the metric could not be computed for that row):')
for c in metric_cols:
    n_nan = int(df[c].isna().sum())
    note = f'  ({n_nan}/{len(df)} rows NaN, averaged over the rest)' if n_nan else ''
    print(f'  {c:20s} {df[c].mean():.4f}{note}')

os.makedirs('evaluation_output', exist_ok=True)
csv_path = 'evaluation_output/ragas_scores.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved per-question scores to {csv_path}')

display_cols = [c for c in ['question'] + metric_cols if c in df.columns]
df[display_cols]

## Step 7: RAGAS compared with Bedrock's own evaluation

RAGAS and the Bedrock Evaluation Job (notebook 02) overlap, but they do not measure quite the same things. It helps to run both.

| Point of comparison | RAGAS (this notebook) | Bedrock Evaluation Job (02) |
|---------------------|-----------------------|------------------------------|
| Where it runs | Locally, in this notebook | As a managed Bedrock job |
| Generation on a Managed KB | Yes, via `AgenticRetrieveStream` | Retrieval only |
| Made-up facts / grounding | `faithfulness` | `Builtin.Faithfulness` (generation only) |
| Answer quality | `answer_relevancy` | `Builtin.Correctness`, `Builtin.Helpfulness` |
| Retrieval quality | `context_precision`, `context_recall` | `Builtin.ContextRelevance`, `Builtin.ContextCoverage` |
| Needs a ground truth | Yes | Optional |
| What you pay for | Your Bedrock model calls | Bedrock evaluation job pricing |

How to read the results: if `context_precision` or `context_recall` is low, work on chunking, the number of results, or reranking. If `faithfulness` is low but the context scores are high, the model is drifting away from the text it retrieved.

## Summary

This notebook scored a Bedrock Managed KB with RAGAS by:

1. Loading a question-and-answer dataset (from notebook 01 or the built-in sample)
2. Generating answers and contexts with `AgenticRetrieveStream`, the path a Managed KB supports
3. Scoring faithfulness, answer relevancy, context precision, and context recall using Bedrock models
4. Saving the per-question scores and comparing them with Bedrock's own evaluation

### What each metric means and how to raise it

| Metric | Range | How to raise it |
|--------|-------|-----------------|
| Faithfulness | 0 to 1 | Prompt the model to stick to the context, lower the temperature, give it better context |
| Answer relevancy | 0 to 1 | Write a clearer generation prompt, retrieve more focused chunks |
| Context precision | 0 to 1 | Add reranking, return fewer but better chunks, filter on metadata |
| Context recall | 0 to 1 | Return more results, use smaller chunks, add chunk overlap |

### Docs

- [RAGAS documentation](https://docs.ragas.io/)
- [AgenticRetrieveStream for Managed KBs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed.html)
- [Bedrock Evaluation Jobs](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-kb.html)
- See `02-bedrock-evaluation-job.ipynb` and `03-agentcore-evaluation-for-managed-kb.ipynb` for the other two approaches